#Changing HTML to String

In [ ]:
!pip install ftfy imbalanced-learn pingouin nltk textstat
import html
import re
import ftfy
import nltk
import numpy as np
import pandas as pd
import textstat
import torch
from google.colab import files
from scipy.sparse import csr_matrix, hstack
from scipy.stats import loguniform
from sklearn.decomposition import NMF, TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, cross_val_score, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer
from imblearn.over_sampling import RandomOverSampler
from nltk import pos_tag
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import sent_tokenize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.6 MB/s eta 0:00:00


In [ ]:
# -----------------------------
# Global config (edit here)
# -----------------------------
RANDOM_STATE = 42

# Data/file settings
DATA_FILENAME = 'all_tasks_90_sub_23_12.csv'
NOTEBOOK_RUN_MODE = 'one_shot_masked_only'

# Tokenizers/encoders used in this notebook
TOKENIZER_MODEL_NAMES = {
    'longformer': 'allenai/longformer-base-4096',
    'bigbird': 'google/bigbird-roberta-base',
}

print('Config loaded')
print('RANDOM_STATE =', RANDOM_STATE)
print('DATA_FILENAME =', DATA_FILENAME)
print('MODE =', NOTEBOOK_RUN_MODE)


In [ ]:
files.upload()
import pandas as pd

df = pd.read_csv("all_tasks_90_sub_23_12.csv", encoding="utf-8", encoding_errors="replace")
df = df.applymap(
    lambda x: ftfy.fix_text(x) if isinstance(x, str) else x
)
df.head()


Saving all_tasks_90_sub_23_12.csv to all_tasks_90_sub_23_12.csv


/tmp/ipykernel_621/2828561421.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(


,msg_id,gpt_interface_id,participant_id,unix_time_when_msg_sent_to_gpt,date_time_when_msg_sent_to_gpt,unix_time_when_msg_received_from_gpt,date_time_when_msg_received_from_gpt,message_to_gpt,message_from_gpt,msg_count_within_p,TASK,subject_id,subject_group,Sex,Age,response_time_sec,response_time_min,words_in_message_to_gpt
0,39,76LD7HTG336W7YD3,JATPTNTXWN4NJ4E3PBR4FRALRG6CW6,1.758823e+09,NaN,1.758823e+09,NaN,Where is best to stay in andora,"Andorra, a small principality nestled between ...",1,trip_task,62e15cc80e4bd2ad93f10f5e,Young_Adults,Male,31,7.9523,0.132538,7
1,40,76LD7HTG336W7YD3,JATPTNTXWN4NJ4E3PBR4FRALRG6CW6,1.758823e+09,NaN,1.758823e+09,NaN,What activities is there to do on a low budget,There are plenty of activities you can enjoy o...,2,trip_task,62e15cc80e4bd2ad93f10f5e,Young_Adults,Male,31,10.7440,0.179067,10
2,41,76LD7HTG336W7YD3,JATPTNTXWN4NJ4E3PBR4FRALRG6CW6,1.758823e+09,NaN,1.758823e+09,NaN,Where is best to eat in Andora,Andorra offers a diverse dining scene that cat...,3,trip_task,62e15cc80e4bd2ad93f10f5e,Young_Adults,Male,31,16.4215,0.273692,7
3,42,76LD7HTG336W7YD3,JATPTNTXWN4NJ4E3PBR4FRALRG6CW6,1.758823e+09,NaN,1.758823e+09,NaN,Is there beaches to visit in andora,Andorra is a landlocked country located in the...,4,trip_task,62e15cc80e4bd2ad93f10f5e,Young_Adults,Male,31,8.7013,0.145022,7
4,43,76LD7HTG336W7YD3,JATPTNTXWN4NJ4E3PBR4FRALRG6CW6,1.758823e+09,NaN,1.758823e+09,NaN,What is the temperature like in andorra,Andorra's climate is characterized by its moun...,5,trip_task,62e15cc80e4bd2ad93f10f5e,Young_Adults,Male,31,13.2633,0.221055,7


#Age Masking

logic is like this:
1) number in words (example: "thirty five")
2) patterns
3) birthday contex

according to 1+2+3 we can mask a string
loging section is for debugging and works only locally (not here in colab)




WHAT YOU NEED TO DO:
1) upload a file called "data.csv"
2) run the following cell

In [ ]:
import re

# -----------------------
# Number patterns
# -----------------------
NUMBER_WORDS = (
    r"(?:one|two|three|four|five|six|seven|eight|nine|ten|"
    r"eleven|twelve|thirteen|fourteen|fifteen|sixteen|"
    r"seventeen|eighteen|nineteen|twenty|thirty|forty|"
    r"fifty|sixty|seventy|eighty|ninety|hundred)"
)

NUMBER_PATTERN = fr"(?:\d+|{NUMBER_WORDS}(?:[- ]{NUMBER_WORDS})?)"

# -----------------------
# Strong patterns (always applied)
# group_idx can be int or tuple of ints
# -----------------------
PATTERNS = [

    # X years old / X-year-old / typos
    (re.compile(
        fr"\b({NUMBER_PATTERN})(\s*[- ]?(?:years?|yrs?|tear|yeat)[-\s]+old)\b",
        re.IGNORECASE
    ), 1),

    # X yo / y.o.
    (re.compile(
        fr"\b({NUMBER_PATTERN})(\s*[- ]?(?:y\.?o\.?|y/o))\b",
        re.IGNORECASE
    ), 1),

    # aged X
    (re.compile(
        fr"\b(aged?\s+)({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), 2),

    # aged X to Y / aged X-Y
    (re.compile(
        fr"\b(aged?\s+)({NUMBER_PATTERN})\s*(?:to|-|–)\s*({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), (2, 3)),

    # turning X
    (re.compile(
        fr"\b(turning\s+)({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), 2),

    # Xth birthday
    (re.compile(
        fr"\b({NUMBER_PATTERN})(?:st|nd|rd|th)?(\s+birt?h?day)\b",
        re.IGNORECASE
    ), 1),

    # will be X (filtered later by context)
    (re.compile(
        fr"\b(will\s+be\s+)({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), 2),

    # in his/her/their/my 20s
    (re.compile(
        fr"\b(in\s+(?:her|his|their|my|our)\s+)({NUMBER_PATTERN})('?s)\b",
        re.IGNORECASE
    ), 2),

    # in their early/mid/late 20s
    (re.compile(
        fr"\b(in\s+their\s+(?:early|mid|late)\s+)({NUMBER_PATTERN})('?s)\b",
        re.IGNORECASE
    ), 2),
]

# -----------------------
# Weak patterns (require age context)
# -----------------------
WEAK_PATTERNS = [

    # is/was X
    (re.compile(
        fr"\b((?:is|was)\s+)({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), 2),

    # I am / I'm X
    (re.compile(
        fr"\b((?:I\s+am|I'm)\s+)({NUMBER_PATTERN})\b",
        re.IGNORECASE
    ), 2),
]

# -----------------------
# Masking function
# -----------------------
def mask_text(text: str, mask_str: str = "###") -> str:
    if not text:
        return text

    masked_text = text

    # Context required for weak patterns
    has_context = re.search(
        r"\b(birt?h?day|bday|card|greeting|turning|years?\s+old|aged?|age|"
        r"son|daughter|child|grandson|granddaughter)\b",
        text,
        re.IGNORECASE
    )

    active_patterns = PATTERNS + (WEAK_PATTERNS if has_context else [])

    for pattern, group_idx in active_patterns:

        def replacer(match):
            full = match.group(0)

            # support single group or multiple groups
            group_idxs = (
                group_idx if isinstance(group_idx, tuple) else (group_idx,)
            )

            # replace from right to left to keep indices valid
            for gi in sorted(group_idxs, reverse=True):
                g_start = match.start(gi) - match.start(0)
                g_end = match.end(gi) - match.start(0)
                full = full[:g_start] + mask_str + full[g_end:]

            return full

        masked_text = pattern.sub(replacer, masked_text)

    return masked_text



In [ ]:
for col in ['message_to_gpt', 'message_from_gpt']:
    if col in df.columns:
        df[f'{col}_age_masked'] = df[col].apply(
            lambda x: mask_text(x) if isinstance(x, str) else x
        )

In [ ]:
df['age_masked_to_gpt'] = (
    df['message_to_gpt'] != df['message_to_gpt_age_masked']
)

df['age_masked_from_gpt'] = (
    df['message_from_gpt'] != df['message_from_gpt_age_masked']
)

print('Masked user messages:', int(df['age_masked_to_gpt'].sum()))
print('Masked assistant messages:', int(df['age_masked_from_gpt'].sum()))
print('One-shot mode: no intermediate CSV is exported.')

In [ ]:
df['age_masked_from_gpt'].sum()
df['age_masked_to_gpt'].sum()

np.int64(42)

In [ ]:
cols_to_keep = [
    'gpt_interface_id',
    'unix_time_when_msg_sent_to_gpt',
    'unix_time_when_msg_received_from_gpt',
    'msg_count_within_p',
    'TASK',
    'subject_id',
    'message_to_gpt',
    'message_from_gpt',
    'message_to_gpt_age_masked',
    'message_from_gpt_age_masked',
    'subject_group',
    'Sex',
    'Age']

cols_to_keep = [c for c in cols_to_keep if c in df.columns]


In [ ]:
df_greeting = df[df['TASK'] == 'greeting_task'].copy()
df_trip = df[df['TASK'] == 'trip_task'].copy()
df_practice = df[df['TASK'] == 'practice_task'].copy()
df_greeting = df_greeting[cols_to_keep].copy()
df_trip = df_trip[cols_to_keep].copy()
df_practice = df_practice[cols_to_keep].copy()

In [ ]:
df_trip = (
    df_trip
    .sort_values(
        by=['subject_id', 'unix_time_when_msg_sent_to_gpt']
    )
    .assign(
        turn_index=lambda x: x.groupby('subject_id').cumcount() + 1
    )
)


In [ ]:
df_greeting = (
    df_greeting
    .sort_values(
        by=['subject_id', 'unix_time_when_msg_sent_to_gpt']
    )
    .assign(
        turn_index=lambda x: x.groupby('subject_id').cumcount() + 1
    )
)

df_practice = (
    df_practice
    .sort_values(
        by=['subject_id', 'unix_time_when_msg_sent_to_gpt']
    )
    .assign(
        turn_index=lambda x: x.groupby('subject_id').cumcount() + 1
    )
)


In [ ]:
df_trip

,unix_time_when_msg_sent_to_gpt,unix_time_when_msg_received_from_gpt,msg_count_within_p,TASK,subject_id,message_to_gpt_age_masked,message_from_gpt_age_masked,subject_group,Sex,Age,turn_index
186,1.761598e+09,1.761598e+09,1,trip_task,568a9909316b10000c50ae46,I would like to spend a four day holiday in An...,Andorra is a small but beautiful country nestl...,Older_Adults,Female,75,1
187,1.761598e+09,1.761598e+09,2,trip_task,568a9909316b10000c50ae46,I am in my seventies. I would like to hike but...,Certainly! Here's a revised itinerary for your...,Older_Adults,Female,75,2
277,1.762889e+09,1.762889e+09,1,trip_task,56b12c0e9f1826000cbcc76c,I want you to help me plan a 4 day trip to And...,Planning a budget-friendly trip from the UK to...,Young_Adults,Female,37,1
278,1.762890e+09,1.762890e+09,2,trip_task,56b12c0e9f1826000cbcc76c,thank you. What activities are there to do in ...,Andorra offers a variety of activities for vis...,Young_Adults,Female,37,2
279,1.762890e+09,1.762890e+09,3,trip_task,56b12c0e9f1826000cbcc76c,does Andorra have any interesting history?,"Yes, Andorra has a fascinating history, shaped...",Young_Adults,Female,37,3
...,...,...,...,...,...,...,...,...,...,...,...
41,1.759848e+09,1.759848e+09,2,trip_task,6783a1eee721ad7b19a43435,"I am not too keen on outdoor activities , can ...","Of course, I can adjust the itinerary to focus...",Older_Adults,Female,66,2
42,1.759848e+09,1.759848e+09,3,trip_task,6783a1eee721ad7b19a43435,can you tell me more about the best local rest...,Certainly! Here is some information on the bes...,Older_Adults,Female,66,3
43,1.759848e+09,1.759848e+09,4,trip_task,6783a1eee721ad7b19a43435,I always like to take gifts back home for frie...,When considering gifts to bring back from Ando...,Older_Adults,Female,66,4
44,1.759848e+09,1.759848e+09,5,trip_task,6783a1eee721ad7b19a43435,Would you recommend the spa products,"Yes, I would recommend considering the spa pro...",Older_Adults,Female,66,5


In [ ]:
df_trip['order_mismatch'] = (
    df_trip['turn_index'] != df_trip['msg_count_within_p']
)
df_trip['order_mismatch'].value_counts()


,count
order_mismatch,
False,435
True,9


In [ ]:
df_trip.loc[
    df_trip['order_mismatch'],
    ['subject_id',
     'unix_time_when_msg_sent_to_gpt',
     'msg_count_within_p',
     'turn_index',
     'message_to_gpt_age_masked',
     'subject_group',
     'Age']
].sort_values(['subject_id', 'turn_index'])


,subject_id,unix_time_when_msg_sent_to_gpt,msg_count_within_p,turn_index,message_to_gpt_age_masked,subject_group,Age
299,5827634ea80bf4000199994b,1.762897e+09,1,2,"plan a trip to Andorra for me. Four days, thre...",Older_Adults,68
431,5ec1cb44516ced000af5f279,1.765211e+09,1,2,I am travelling to Andorra in September for a ...,Young_Adults,30
162,5f836c9938d2da4fb75631ef,1.761592e+09,3,4,what about passports and customs controls? Am ...,Older_Adults,69
183,649d81f47df8018a01c555dd,1.761596e+09,2,3,Thanks. We want to stop somewhere quite quiet ...,Older_Adults,70
432,6674a611c78207e9009eaadd,1.765211e+09,1,2,planning a trip to benidorm,Young_Adults,35
201,66a90e7caa371f25183bdce2,1.761680e+09,5,6,WHich reviews do you look at?,Older_Adults,72
49,673dbbee9983e0d8084c8a76,1.759849e+09,2,3,I am only going for 4-days. How should I fill ...,Older_Adults,69
50,673dbbee9983e0d8084c8a76,1.759849e+09,2,4,I am only going for 4-days. How should I fill ...,Older_Adults,69
51,673dbbee9983e0d8084c8a76,1.759849e+09,2,5,I am only going for 4-days. How should I fill ...,Older_Adults,69


In [ ]:
df_greeting['order_mismatch'] = (
    df_greeting['turn_index'] != df_greeting['msg_count_within_p']
)

df_greeting['order_mismatch'].value_counts()


,count
order_mismatch,
False,450
True,5


In [ ]:
df_greeting.loc[
    df_greeting['order_mismatch'],
    ['subject_id',
     'unix_time_when_msg_sent_to_gpt',
     'msg_count_within_p',
     'turn_index',
     'message_to_gpt_age_masked',
     'subject_group',
     'Age']
].sort_values(['subject_id', 'turn_index'])


,subject_id,unix_time_when_msg_sent_to_gpt,msg_count_within_p,turn_index,message_to_gpt_age_masked,subject_group,Age
985,66e4906547365a7208341b04,1.761593e+09,1,2,Can you create a birthday message for my husba...,Young_Adults,34
984,66e4906547365a7208341b04,1.761593e+09,1,3,Can you create a birthday message for my husba...,Young_Adults,34
987,66e4906547365a7208341b04,1.761593e+09,1,4,Can you create a birthday message for my husba...,Young_Adults,34
1238,671f5208a60be0a55c28af8c,1.765125e+09,5,6,We live near the beach on Hayling Island. What...,Young_Adults,38
1240,671f5208a60be0a55c28af8c,1.765125e+09,6,7,What is a good poem that reflects my husband's...,Young_Adults,38


In [ ]:
# ------------------------------------------------
# Helper function: build a full conversation transcript
# ------------------------------------------------
def build_full_conversation(df_chat: pd.DataFrame) -> str:
    """
    Build a full conversation transcript for one chat using age-masked text.
    """
    user_col = 'message_to_gpt_age_masked'
    assistant_col = 'message_from_gpt_age_masked'

    conversation_turns = []
    for _, row in df_chat.iterrows():
        user_msg = row.get(user_col, '')
        assistant_msg = row.get(assistant_col, '')

        if isinstance(user_msg, str) and user_msg.strip():
            conversation_turns.append(f"[user] {user_msg}")
        if isinstance(assistant_msg, str) and assistant_msg.strip():
            conversation_turns.append(f"[assistant] {assistant_msg}")

    return "\n\n".join(conversation_turns)


# ------------------------------------------------
# Helper function: build chat-level dataframe
# ------------------------------------------------
def build_conversation_df(df_task: pd.DataFrame) -> pd.DataFrame:
    """
    Mimic original notebook behavior: one conversation per subject.
    This is stable across exports even when gpt_interface_id is reused.
    """
    sort_cols = [c for c in ['subject_id', 'turn_index', 'msg_count_within_p', 'unix_time_when_msg_sent_to_gpt'] if c in df_task.columns]
    df_task = df_task.sort_values(sort_cols).reset_index(drop=True)

    conv_rows = []
    for subject_id, g in df_task.groupby('subject_id', sort=False, dropna=False):
        conv_rows.append({
            'subject_id': subject_id,
            'full_conversation': build_full_conversation(g)
        })

    df_conversations = pd.DataFrame(conv_rows)
    meta_cols = [c for c in ['subject_id', 'TASK', 'subject_group', 'Age', 'Sex'] if c in df_task.columns]
    meta_per_subject = df_task[meta_cols].drop_duplicates(subset='subject_id')
    df_conversations = df_conversations.merge(meta_per_subject, on='subject_id', how='left')
    df_conversations['chat_id'] = df_conversations['subject_id'].astype(str)
    return df_conversations


# ------------------------------------------------
# Build conversation-level dataframes
# ------------------------------------------------

df_conversations_trip = build_conversation_df(df_trip)
df_conversations_greeting = build_conversation_df(df_greeting)
df_conversations_practice = build_conversation_df(df_practice)


# ------------------------------------------------
# Quick sanity checks
# ------------------------------------------------
print("Trip conversations:", df_conversations_trip.shape[0])
print("Greeting conversations:", df_conversations_greeting.shape[0])
print("Practice conversations:", df_conversations_practice.shape[0])

print("\nExample Trip conversation:\n")
print(df_conversations_trip['full_conversation'].iloc[0][:1000])

#print("\nExample Greeting conversation:\n")
#print(df_conversations_greeting['full_conversation'].iloc[0][:1000])


Trip conversations: 90
Greeting conversations: 90

Example Trip conversation:

[user] I would like to spend a four day holiday in Andorra travelling on my own. I like staying in hostels, preferably YHA hostels. Can you suggest where I can stay and the best places to visit in Andorra. Thank you

[assistant] Andorra is a small but beautiful country nestled in the Pyrenees between France and Spain. It is known for its ski resorts, hiking trails, and tax-free shopping. Here's a guide to help you plan your four-day solo trip to Andorra, focusing on hostel stays and key attractions.

1. **Accommodation:**
   - While Andorra doesn't have YHA hostels specifically, it does offer various budget-friendly accommodations that cater to solo travelers.
   - Consider staying at Alberg la Comella, a hostel that provides comfortable lodging and is conveniently located for accessing Andorra la Vella, the capital city.
   - Another option is Barri Antic Hostel & Pub, situated in the heart of Andorra la Ve

/tmp/ipykernel_621/1283483916.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_full_conversation)
/tmp/ipykernel_621/1283483916.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(build_full_conversation)


In [ ]:
def show_random_conversation(df, label):
    row = df.sample(1, random_state=RANDOM_STATE).iloc[0]
    print(f"--- {label} ---")
    print(f"subject_id: {row.get('subject_id')}")
    print(f"subject_group: {row.get('subject_group')}")
    print(f"Age: {row.get('Age')}")
    print("\nConversation:\n")
    print(row.get('full_conversation', ''))
    print()

show_random_conversation(df_conversations_trip, "Trip task (masked)")
show_random_conversation(df_conversations_greeting, "Greeting task (masked)")
show_random_conversation(df_conversations_practice, "Practice task (masked)")

In [ ]:
df_conversations_trip

,subject_id,full_conversation,TASK,subject_group,Age,word_count,approx_tokens,tokens_longformer,tokens_roberta,longformer_excess
0,568a9909316b10000c50ae46,[user] I would like to spend a four day holida...,trip_task,Older_Adults,75,978,1271,1488,1488,0
1,56b12c0e9f1826000cbcc76c,[user] I want you to help me plan a 4 day trip...,trip_task,Young_Adults,37,1261,1639,2031,2031,0
2,56f699e876348f000c883bba,[user] Please can you help me plan a trip to A...,trip_task,Young_Adults,30,1870,2431,2839,2839,0
3,57698d52688a5c00018d958d,[user] I am trying to plan a 4 day trip to And...,trip_task,Older_Adults,65,2539,3300,3930,3930,0
4,578bb26f6cc44500010448c8,[user] I'd like some help in planning an itine...,trip_task,Older_Adults,70,1694,2202,2612,2612,0
...,...,...,...,...,...,...,...,...,...,...
85,6725c4a8e3ab0787551acf38,[user] i want to plan a trip for 2 adults to a...,trip_task,Older_Adults,65,1563,2031,2346,2346,0
86,673dbbee9983e0d8084c8a76,"[user] what are the ""must-see"" sights in Andor...",trip_task,Older_Adults,69,2975,3867,4606,4606,510
87,6764260cc5b6436f25820b52,[user] I am going on a 4 day trip to Andorra. ...,trip_task,Young_Adults,31,476,618,763,763,0
88,6772e76feabb30f82fcd8f59,[user] I need to make an itinerary for a 4-day...,trip_task,Young_Adults,38,3217,4182,4841,4841,745


In [ ]:
df_conversations_trip['word_count'] = (
    df_conversations_trip['full_conversation']
    .str.split()
    .str.len()
)
df_conversations_trip['word_count'].describe()

,word_count
count,90.000000
mean,1682.922222
std,645.583444
min,476.000000
25%,1245.000000
50%,1558.000000
75%,2206.000000
max,3217.000000


In [ ]:
df_conversations_greeting['word_count'] = (
    df_conversations_greeting['full_conversation']
    .str.split()
    .str.len()
)
df_conversations_greeting['word_count'].describe()

,word_count
count,90.000000
mean,649.833333
std,376.036403
min,150.000000
25%,356.250000
50%,575.500000
75%,828.500000
max,2261.000000


In [ ]:
df_conversations_trip['approx_tokens'] = (df_conversations_trip['word_count'] * 1.3).astype(int)

(df_conversations_trip['approx_tokens'] > 4096).sum()

np.int64(2)

In [ ]:
df_conversations_trip.sort_values('approx_tokens', ascending=False).head(10)
df_conversations_trip.sort_values('approx_tokens', ascending=False)[
    ['subject_id', 'word_count', 'approx_tokens']
].head(10)

,subject_id,word_count,approx_tokens
88,6772e76feabb30f82fcd8f59,3217,4182
73,665f3e69f83003c1df913675,3176,4128
58,6144d8d264b28a29e10ad3de,3055,3971
86,673dbbee9983e0d8084c8a76,2975,3867
84,672107688fa36a5213347a7a,2845,3698
83,671f5208a60be0a55c28af8c,2677,3480
10,59d949cf8fd07a000184247e,2577,3350
12,5ab69c4ba3ba7a0001b405a6,2571,3342
3,57698d52688a5c00018d958d,2539,3300
60,62e15cc80e4bd2ad93f10f5e,2531,3290


In [ ]:
import pandas as pd
from transformers import AutoTokenizer

# Tokenizers for comparison (Longformer + BigBird)
TOKENIZER_SPECS = {
    'longformer': ('allenai/longformer-base-4096', 4096),
    'bigbird': ('google/bigbird-roberta-base', 4096),
}

tokenizers = {name: AutoTokenizer.from_pretrained(model_name) for name, (model_name, _) in TOKENIZER_SPECS.items()}

def count_tokens(text, tokenizer):
    if pd.isna(text):
        return 0
    return len(tokenizer.encode(str(text), add_special_tokens=True, truncation=False))

for split_name, df_conv in {
    'trip': df_conversations_trip,
    'greeting': df_conversations_greeting,
    'practice': df_conversations_practice,
}.items():
    for tok_name, tok in tokenizers.items():
        col = f'tokens_{tok_name}'
        df_conv[col] = df_conv['full_conversation'].apply(lambda x: count_tokens(x, tok))

print('Token counting finished for trip + greeting + practice chat-level datasets.')
for tok_name, (_, limit) in TOKENIZER_SPECS.items():
    trip_over = int((df_conversations_trip[f'tokens_{tok_name}'] > limit).sum())
    greeting_over = int((df_conversations_greeting[f'tokens_{tok_name}'] > limit).sum())
    practice_over = int((df_conversations_practice[f'tokens_{tok_name}'] > limit).sum())
    print(f"{tok_name}: trip>{limit}={trip_over}, greeting>{limit}={greeting_over}, practice>{limit}={practice_over}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (1488 > 512). Running this sequence through the model will result in indexing errors


Trip > 4096 (Longformer): 5
Trip > 512 (RoBERTa): 90
Greeting > 4096 (Longformer): 0
Greeting > 512 (RoBERTa): 68


In [ ]:
LONGFORMER_MAX = 4096

df_conversations_trip['longformer_excess'] = (
    df_conversations_trip['tokens_longformer'] - LONGFORMER_MAX
).clip(lower=0)

In [ ]:
df_conversations_trip.loc[
    df_conversations_trip['longformer_excess'] > 0,
    ['subject_id', 'tokens_longformer', 'longformer_excess', 'subject_group']
].sort_values('longformer_excess', ascending=False)

,subject_id,tokens_longformer,longformer_excess,subject_group
73,665f3e69f83003c1df913675,5217,1121,Young_Adults
88,6772e76feabb30f82fcd8f59,4841,745,Young_Adults
86,673dbbee9983e0d8084c8a76,4606,510,Older_Adults
58,6144d8d264b28a29e10ad3de,4565,469,Young_Adults
84,672107688fa36a5213347a7a,4201,105,Young_Adults


## 5) Feature engineering

In [ ]:
nltk.download('vader_lexicon', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

df_model = pd.concat(
    [df_conversations_trip, df_conversations_greeting, df_conversations_practice],
    ignore_index=True
)

def _split_conversation_roles(text: str):
    if not isinstance(text, str):
        text = ''
    user_blocks = re.findall(r"\[user\]\s*(.*?)(?=\n\n\[assistant\]|\Z)", text, flags=re.S)
    assistant_blocks = re.findall(r"\[assistant\]\s*(.*?)(?=\n\n\[user\]|\Z)", text, flags=re.S)
    return ' '.join(user_blocks).strip(), ' '.join(assistant_blocks).strip()

split_parts = df_model['full_conversation'].fillna('').apply(_split_conversation_roles)
df_model['message_to_gpt'] = split_parts.apply(lambda x: x[0])
df_model['message_from_gpt'] = split_parts.apply(lambda x: x[1])
text_to_gpt = df_model['message_to_gpt'].fillna('')
text_from_gpt = df_model['message_from_gpt'].fillna('')


### 5.1 Text length features

In [ ]:
df_model['msg_to_gpt_char_count'] = text_to_gpt.str.len()
df_model['msg_to_gpt_word_count'] = text_to_gpt.str.split().str.len()
df_model['msg_from_gpt_char_count'] = text_from_gpt.str.len()
df_model['msg_from_gpt_word_count'] = text_from_gpt.str.split().str.len()
df_model['msg_to_gpt_avg_word_len'] = df_model['msg_to_gpt_char_count'] / (df_model['msg_to_gpt_word_count'] + 1)
df_model['msg_from_gpt_avg_word_len'] = df_model['msg_from_gpt_char_count'] / (df_model['msg_from_gpt_word_count'] + 1)
df_model['response_word_ratio'] = df_model['msg_from_gpt_word_count'] / (df_model['msg_to_gpt_word_count'] + 1)


### 5.2 Sentiment features (VADER)

In [ ]:
sid = SentimentIntensityAnalyzer()
to_scores = text_to_gpt.apply(lambda txt: sid.polarity_scores(str(txt)))
from_scores = text_from_gpt.apply(lambda txt: sid.polarity_scores(str(txt)))

df_model['msg_to_gpt_sentiment_compound'] = to_scores.apply(lambda s: s['compound'])
df_model['msg_to_gpt_sentiment_pos'] = to_scores.apply(lambda s: s['pos'])
df_model['msg_to_gpt_sentiment_neg'] = to_scores.apply(lambda s: s['neg'])
df_model['msg_from_gpt_sentiment_compound'] = from_scores.apply(lambda s: s['compound'])
df_model['msg_from_gpt_sentiment_pos'] = from_scores.apply(lambda s: s['pos'])
df_model['msg_from_gpt_sentiment_neg'] = from_scores.apply(lambda s: s['neg'])


### 5.3 Readability features

In [ ]:
df_model['msg_to_gpt_flesch_kincaid'] = text_to_gpt.apply(textstat.flesch_kincaid_grade)
df_model['msg_to_gpt_flesch_ease'] = text_to_gpt.apply(textstat.flesch_reading_ease)
df_model['msg_to_gpt_ari'] = text_to_gpt.apply(textstat.automated_readability_index)
df_model['msg_from_gpt_flesch_kincaid'] = text_from_gpt.apply(textstat.flesch_kincaid_grade)
df_model['msg_from_gpt_flesch_ease'] = text_from_gpt.apply(textstat.flesch_reading_ease)


### 5.4 TF-IDF + SVD features

In [ ]:
def _safe_svd_components(matrix, cap=20):
    return max(1, min(cap, matrix.shape[0] - 1, matrix.shape[1] - 1))

tfidf_to = TfidfVectorizer(ngram_range=(1, 2), max_features=500, min_df=2, stop_words='english')
tfidf_to_matrix = tfidf_to.fit_transform(text_to_gpt)
n_svd_to = _safe_svd_components(tfidf_to_matrix, cap=20)
svd_to = TruncatedSVD(n_components=n_svd_to, random_state=RANDOM_STATE)
tfidf_to_svd = svd_to.fit_transform(tfidf_to_matrix)
tfidf_to_df = pd.DataFrame(tfidf_to_svd, columns=[f'tfidf_to_svd_{i}' for i in range(n_svd_to)], index=df_model.index)

tfidf_from = TfidfVectorizer(ngram_range=(1, 2), max_features=500, min_df=2, stop_words='english')
tfidf_from_matrix = tfidf_from.fit_transform(text_from_gpt)
n_svd_from = _safe_svd_components(tfidf_from_matrix, cap=20)
svd_from = TruncatedSVD(n_components=n_svd_from, random_state=RANDOM_STATE)
tfidf_from_svd = svd_from.fit_transform(tfidf_from_matrix)
tfidf_from_df = pd.DataFrame(tfidf_from_svd, columns=[f'tfidf_from_svd_{i}' for i in range(n_svd_from)], index=df_model.index)

df_model = pd.concat([df_model, tfidf_to_df, tfidf_from_df], axis=1)


### 5.5 NMF topic features

In [ ]:
def _safe_nmf_components(matrix, cap=10):
    return max(1, min(cap, matrix.shape[0], matrix.shape[1]))

n_topics_to = _safe_nmf_components(tfidf_to_matrix, cap=10)
n_topics_from = _safe_nmf_components(tfidf_from_matrix, cap=10)

nmf_to = NMF(n_components=n_topics_to, random_state=RANDOM_STATE)
nmf_to_df = pd.DataFrame(nmf_to.fit_transform(tfidf_to_matrix), columns=[f'nmf_to_topic_{i}' for i in range(n_topics_to)], index=df_model.index)

nmf_from = NMF(n_components=n_topics_from, random_state=RANDOM_STATE)
nmf_from_df = pd.DataFrame(nmf_from.fit_transform(tfidf_from_matrix), columns=[f'nmf_from_topic_{i}' for i in range(n_topics_from)], index=df_model.index)

df_model = pd.concat([df_model, nmf_to_df, nmf_from_df], axis=1)


### 5.6 Lexical and POS-style features

In [ ]:
def _lexical_style_stats(text: str):
    import string
    from collections import Counter

    text = str(text) if text is not None else ''
    words = re.findall(r"\w+", text.lower())
    n_words = len(words)
    n_chars = len(text)

    unique_words = len(set(words))
    counts = Counter(words)
    hapax = sum(1 for _, c in counts.items() if c == 1)
    punct_density = (sum(1 for ch in text if ch in string.punctuation) / n_chars) if n_chars else 0.0

    if text.strip():
        try:
            sentences = sent_tokenize(text)
        except LookupError:
            sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    else:
        sentences = []

    return {
        'ttr': (unique_words / n_words) if n_words else 0.0,
        'hapax_ratio': (hapax / n_words) if n_words else 0.0,
        'punct_density': punct_density,
        'avg_sentence_length': (n_words / len(sentences)) if len(sentences) else 0.0,
        'question_marks': text.count('?'),
        'exclamation_marks': text.count('!'),
    }

def _pos_ratios(text: str):
    words = re.findall(r"\w+", str(text))[:200]
    if not words:
        return {'noun_ratio': 0.0, 'verb_ratio': 0.0, 'adj_ratio': 0.0, 'adv_ratio': 0.0}
    tagged = pos_tag(words)
    n = len(tagged)
    return {
        'noun_ratio': sum(1 for _, t in tagged if t.startswith('NN')) / n,
        'verb_ratio': sum(1 for _, t in tagged if t.startswith('VB')) / n,
        'adj_ratio': sum(1 for _, t in tagged if t.startswith('JJ')) / n,
        'adv_ratio': sum(1 for _, t in tagged if t.startswith('RB')) / n,
    }

lex_to = pd.DataFrame(text_to_gpt.apply(_lexical_style_stats).tolist(), index=df_model.index).add_prefix('msg_to_')
lex_from = pd.DataFrame(text_from_gpt.apply(_lexical_style_stats).tolist(), index=df_model.index).add_prefix('msg_from_')
pos_to = pd.DataFrame(text_to_gpt.apply(_pos_ratios).tolist(), index=df_model.index).add_prefix('msg_to_')
pos_from = pd.DataFrame(text_from_gpt.apply(_pos_ratios).tolist(), index=df_model.index).add_prefix('msg_from_')

df_model = pd.concat([df_model, lex_to, lex_from, pos_to, pos_from], axis=1)
print('Feature engineering complete. df_model shape:', df_model.shape)


In [ ]:
df_model = df_model.copy()

In [ ]:
df_model['target'] = df_model['subject_group'].map({
    'Young_Adults': 0,
    'Older_Adults': 1
})

X = df_model['full_conversation']
y = df_model['target']

In [ ]:
print(df_model.shape)

(180, 11)


In [ ]:
df_model

,subject_id,full_conversation,TASK,subject_group,Age,word_count,approx_tokens,tokens_longformer,tokens_roberta,longformer_excess,target
0,568a9909316b10000c50ae46,[user] I would like to spend a four day holida...,trip_task,Older_Adults,75,978,1271.0,1488,1488,0.0,1
1,56b12c0e9f1826000cbcc76c,[user] I want you to help me plan a 4 day trip...,trip_task,Young_Adults,37,1261,1639.0,2031,2031,0.0,0
2,56f699e876348f000c883bba,[user] Please can you help me plan a trip to A...,trip_task,Young_Adults,30,1870,2431.0,2839,2839,0.0,0
3,57698d52688a5c00018d958d,[user] I am trying to plan a 4 day trip to And...,trip_task,Older_Adults,65,2539,3300.0,3930,3930,0.0,1
4,578bb26f6cc44500010448c8,[user] I'd like some help in planning an itine...,trip_task,Older_Adults,70,1694,2202.0,2612,2612,0.0,1
...,...,...,...,...,...,...,...,...,...,...,...
175,6725c4a8e3ab0787551acf38,[user] hi its my daughters ###th birtday what ...,greeting_task,Older_Adults,65,570,NaN,768,768,NaN,1
176,673dbbee9983e0d8084c8a76,[user] Write a birthday greeting for my old sc...,greeting_task,Older_Adults,69,613,NaN,822,822,NaN,1
177,6764260cc5b6436f25820b52,[user] Can you please help me create a birthda...,greeting_task,Young_Adults,31,242,NaN,384,384,NaN,0
178,6772e76feabb30f82fcd8f59,[user] i want a fun birthday card greeting for...,greeting_task,Young_Adults,38,581,NaN,950,950,NaN,0


In [ ]:
from sklearn.model_selection import train_test_split

def subject_level_train_test_split(df_in, test_size=0.2, random_state=RANDOM_STATE):
    subjects_df_local = (
        df_in[['subject_id', 'subject_group', 'target']]
        .drop_duplicates(subset='subject_id')
        .reset_index(drop=True)
    )
    train_subjects_local, test_subjects_local = train_test_split(
        subjects_df_local,
        test_size=test_size,
        random_state=random_state,
        stratify=subjects_df_local['target'],
    )
    return train_subjects_local.reset_index(drop=True), test_subjects_local.reset_index(drop=True)

# Robust stratified split (same logic style as clean_only_bert notebook)
# Avoids fixed n=9 sampling failures when class counts are smaller.
train_subjects, test_subjects = subject_level_train_test_split(df_model)

In [ ]:
test_ids = test_subjects['subject_id']
train_ids = train_subjects['subject_id']

df_test = df_model[df_model['subject_id'].isin(test_ids)].copy()
df_train = df_model[df_model['subject_id'].isin(train_ids)].copy()

In [ ]:
print("Test rows:", df_test.shape[0])
print("Train rows:", df_train.shape[0])
print("Test subjects:", df_test['subject_id'].nunique())
print("Train subjects:", df_train['subject_id'].nunique())

print("\nTest distribution (subjects):")
print(test_subjects['target'].value_counts().sort_index())
print("\nTrain distribution (subjects):")
print(train_subjects['target'].value_counts().sort_index())

Test rows: 36
Train rows: 144
Test subjects: 18
Train subjects: 72

Test distribution (subjects):
target
0    9
1    9
Name: count, dtype: int64


In [ ]:
X_train = df_train['full_conversation'].tolist()
y_train = df_train['target'].values

X_test = df_test['full_conversation'].tolist()
y_test = df_test['target'].values

In [ ]:
train_meta = df_train[['subject_id', 'TASK', 'subject_group', 'target']].reset_index(drop=True)
test_meta = df_test[['subject_id', 'TASK', 'subject_group', 'target']].reset_index(drop=True)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Running on device:', device)

ENCODER_SPECS = {
    'bigbird': {
        'model_name': 'google/bigbird-roberta-base',
        'max_length': 4096,
        'use_global_attention': False,
    },
}


Loading weights:   0%|          | 0/271 [00:00<?, ?it/s]

LongformerModel LOAD REPORT from: allenai/longformer-base-4096
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.decoder.weight    | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


LongformerModel(
  (embeddings): LongformerEmbeddings(
    (word_embeddings): Embedding(50265, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (position_embeddings): Embedding(4098, 768, padding_idx=1)
  )
  (encoder): LongformerEncoder(
    (layer): ModuleList(
      (0-11): 12 x LongformerLayer(
        (attention): LongformerAttention(
          (self): LongformerSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (query_global): Linear(in_features=768, out_features=768, bias=True)
            (key_global): Linear(in_features=768, out_features=768, bias=True)
            (value_global): Linear(in_features=768, out_features=768, bias=True)
          )
    

## 6) Embedding extraction and classifier (after tokenizer)

This stage converts each chat text into a dense vector (embedding) with a transformer encoder.  
Then a Logistic Regression classifier is trained on those vectors for age-group classification. TF-IDF baselines with meta-features are added below, plus an optional MLP baseline.

In [ ]:
def load_encoder(spec):
    model_name = spec['model_name']
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)
    model.eval()
    return tokenizer, model


def _pool_hidden(last_hidden, attention_mask, pooling):
    if pooling == 'cls':
        return last_hidden[:, 0, :]

    mask = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    if pooling == 'mean':
        masked = last_hidden * mask
        return masked.sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

    if pooling == 'max':
        masked = last_hidden.masked_fill(mask == 0, -1e9)
        return masked.max(dim=1).values

    raise ValueError(f'Unsupported pooling: {pooling}')


def get_embeddings(text_list, tokenizer, model, max_length=512, batch_size=2, use_global_attention=False, pooling='mean'):
    all_embeddings = []
    model_device = next(model.parameters()).device
    with torch.no_grad():
        for i in tqdm(range(0, len(text_list), batch_size), desc='Extracting embeddings', unit='batch'):
            batch_texts = text_list[i:i + batch_size]
            inputs = tokenizer(
                batch_texts,
                return_tensors='pt',
                padding=True,
                truncation=True,
                max_length=max_length,
            )
            inputs = {k: v.to(model_device) for k, v in inputs.items()}

            forward_kwargs = dict(inputs)
            if use_global_attention and 'longformer' in model.__class__.__name__.lower():
                global_attention_mask = torch.zeros_like(inputs['attention_mask'])
                global_attention_mask[:, 0] = 1
                forward_kwargs['global_attention_mask'] = global_attention_mask

            outputs = model(**forward_kwargs)
            pooled = _pool_hidden(outputs.last_hidden_state, inputs['attention_mask'], pooling)
            all_embeddings.append(pooled.detach().cpu().numpy())

    return np.vstack(all_embeddings)


def unload_encoder(model):
    if model is not None:
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


In [ ]:
FEATURE_ENGINEERING_META_COLUMNS = [
    'word_count', 'approx_tokens', 'tokens_longformer', 'tokens_bigbird', 'longformer_excess',
    'msg_to_gpt_char_count', 'msg_to_gpt_word_count', 'msg_from_gpt_char_count', 'msg_from_gpt_word_count',
    'msg_to_gpt_avg_word_len', 'msg_from_gpt_avg_word_len', 'response_word_ratio',
    'msg_to_gpt_sentiment_compound', 'msg_to_gpt_sentiment_pos', 'msg_to_gpt_sentiment_neg',
    'msg_from_gpt_sentiment_compound', 'msg_from_gpt_sentiment_pos', 'msg_from_gpt_sentiment_neg',
    'msg_to_gpt_flesch_kincaid', 'msg_to_gpt_flesch_ease', 'msg_to_gpt_ari',
    'msg_from_gpt_flesch_kincaid', 'msg_from_gpt_flesch_ease',
    'msg_to_ttr', 'msg_to_hapax_ratio', 'msg_to_punct_density', 'msg_to_avg_sentence_length',
    'msg_to_question_marks', 'msg_to_exclamation_marks', 'msg_to_noun_ratio', 'msg_to_verb_ratio', 'msg_to_adj_ratio', 'msg_to_adv_ratio',
    'msg_from_ttr', 'msg_from_hapax_ratio', 'msg_from_punct_density', 'msg_from_avg_sentence_length',
    'msg_from_question_marks', 'msg_from_exclamation_marks', 'msg_from_noun_ratio', 'msg_from_verb_ratio', 'msg_from_adj_ratio', 'msg_from_adv_ratio',
]
FEATURE_ENGINEERING_META_COLUMNS += [
    c for c in df_model.columns
    if c.startswith('tfidf_to_svd_') or c.startswith('tfidf_from_svd_') or c.startswith('nmf_to_topic_') or c.startswith('nmf_from_topic_')
]


def _extract_meta_features(text: str):
    if not isinstance(text, str):
        text = ''

    user_blocks = re.findall(r"\[user\]\s*(.*?)(?=\n\n\[assistant\]|\Z)", text, flags=re.S)
    assistant_blocks = re.findall(r"\[assistant\]\s*(.*?)(?=\n\n\[user\]|\Z)", text, flags=re.S)

    user_chars = sum(len(x) for x in user_blocks)
    assistant_chars = sum(len(x) for x in assistant_blocks)
    total_chars = len(text)
    words = re.findall(r"\w+", text)
    total_words = len(words)
    avg_word_len = (sum(len(w) for w in words) / total_words) if total_words else 0.0

    user_turns = len(user_blocks)
    assistant_turns = len(assistant_blocks)
    user_assistant_ratio = (user_chars + 1) / (assistant_chars + 1)

    return [
        total_chars,
        total_words,
        avg_word_len,
        user_turns,
        assistant_turns,
        user_chars,
        assistant_chars,
        user_assistant_ratio,
    ]


def build_meta_features(df_conversations: pd.DataFrame) -> np.ndarray:
    texts = df_conversations['full_conversation'].fillna('').tolist()
    base_meta = np.array([_extract_meta_features(t) for t in texts], dtype=float)

    available_cols = [c for c in FEATURE_ENGINEERING_META_COLUMNS if c in df_conversations.columns]
    if not available_cols:
        return base_meta

    engineered_meta = (
        df_conversations[available_cols]
        .apply(pd.to_numeric, errors='coerce')
        .fillna(0.0)
        .to_numpy(dtype=float)
    )
    return np.hstack([base_meta, engineered_meta])


def build_tfidf_features(train_texts, test_texts, train_meta, test_meta):
    from sklearn.feature_extraction.text import TfidfVectorizer
    word_vect = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000)
    char_vect = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), min_df=2, max_features=30000)

    X_train_word = word_vect.fit_transform(train_texts)
    X_test_word = word_vect.transform(test_texts)

    X_train_char = char_vect.fit_transform(train_texts)
    X_test_char = char_vect.transform(test_texts)

    meta_scaler = StandardScaler(with_mean=False)
    X_train_meta = meta_scaler.fit_transform(csr_matrix(train_meta))
    X_test_meta = meta_scaler.transform(csr_matrix(test_meta))

    X_train = hstack([X_train_word, X_train_char, X_train_meta]).tocsr()
    X_test = hstack([X_test_word, X_test_char, X_test_meta]).tocsr()
    return X_train, X_test


def build_bow_features(train_texts, test_texts, train_meta, test_meta):
    from sklearn.feature_extraction.text import CountVectorizer
    word_vect = CountVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000)
    char_vect = CountVectorizer(analyzer='char', ngram_range=(3, 5), min_df=2, max_features=30000)

    X_train_word = word_vect.fit_transform(train_texts)
    X_test_word = word_vect.transform(test_texts)

    X_train_char = char_vect.fit_transform(train_texts)
    X_test_char = char_vect.transform(test_texts)

    meta_scaler = StandardScaler(with_mean=False)
    X_train_meta = meta_scaler.fit_transform(csr_matrix(train_meta))
    X_test_meta = meta_scaler.transform(csr_matrix(test_meta))

    X_train = hstack([X_train_word, X_train_char, X_train_meta]).tocsr()
    X_test = hstack([X_test_word, X_test_char, X_test_meta]).tocsr()
    return X_train, X_test


## 7) One-shot run: masked only + multi-tokenizer comparison

In [ ]:
EMBEDDING_CACHE = {}

def _embedding_cache_key(spec, data_version, split):
    return (
        spec['encoder'],
        spec['max_length'],
        spec['use_global_attention'],
        spec.get('pooling', 'mean'),
        data_version,
        split,
    )


def get_cached_embeddings(df_train_local, df_test_local, spec, data_version):
    key_train = _embedding_cache_key(spec, data_version, 'train')
    key_test = _embedding_cache_key(spec, data_version, 'test')

    X_train_local = EMBEDDING_CACHE.get(key_train)
    X_test_local = EMBEDDING_CACHE.get(key_test)

    if X_train_local is None or X_test_local is None:
        tokenizer, model = load_encoder(spec)
        if X_train_local is None:
            X_train_local = get_embeddings(
                df_train_local['full_conversation'].tolist(),
                tokenizer=tokenizer,
                model=model,
                max_length=spec['max_length'],
                batch_size=2,
                use_global_attention=spec['use_global_attention'],
                pooling=spec.get('pooling', 'mean'),
            )
            EMBEDDING_CACHE[key_train] = X_train_local

        if X_test_local is None:
            X_test_local = get_embeddings(
                df_test_local['full_conversation'].tolist(),
                tokenizer=tokenizer,
                model=model,
                max_length=spec['max_length'],
                batch_size=2,
                use_global_attention=spec['use_global_attention'],
                pooling=spec.get('pooling', 'mean'),
            )
            EMBEDDING_CACHE[key_test] = X_test_local

        unload_encoder(model)

    return X_train_local, X_test_local


def make_group_cv(groups, max_splits=3):
    n_unique_groups = pd.Series(groups).nunique()
    n_splits = min(max_splits, n_unique_groups)
    if n_splits < 2:
        raise ValueError('Need at least 2 unique subjects for grouped CV.')
    return GroupKFold(n_splits=n_splits)


def run_single_experiment(df_model_input, encoder_name, spec, train_subject_ids, test_subject_ids, data_version):
    # Filter using the already fixed subject split; no new split happens here.
    df_train_local = df_model_input[df_model_input['subject_id'].isin(train_subject_ids)].copy()
    df_test_local = df_model_input[df_model_input['subject_id'].isin(test_subject_ids)].copy()

    X_train_local, X_test_local = get_cached_embeddings(
        df_train_local,
        df_test_local,
        spec,
        data_version,
    )

    y_train_local = df_train_local['target'].values
    y_test_local = df_test_local['target'].values

    # Additive mimic from clean_only_bert: CV + light LR tuning
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_local)
    X_test_scaled = scaler.transform(X_test_local)

    train_groups = df_train_local['subject_id'].values
    results = []
    cv = make_group_cv(train_groups, max_splits=3)
    base_lr = LogisticRegression(random_state=RANDOM_STATE, max_iter=3000, class_weight='balanced')
    cv_f1_scores = cross_val_score(base_lr, X_train_scaled, y_train_local, cv=cv, groups=train_groups, scoring='f1')

    param_dist_lr = [
        {'C': loguniform(0.001, 100), 'solver': ['liblinear'], 'penalty': ['l1', 'l2']},
        {'C': loguniform(0.001, 100), 'solver': ['lbfgs'], 'penalty': ['l2']},
    ]
    search_cv = list(make_group_cv(train_groups, max_splits=2).split(X_train_scaled, y_train_local, train_groups))
    lr_search = RandomizedSearchCV(
        estimator=base_lr,
        param_distributions=param_dist_lr,
        n_iter=8,
        scoring='f1',
        cv=search_cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0,
    )
    lr_search.fit(X_train_scaled, y_train_local)
    y_pred_local = lr_search.best_estimator_.predict(X_test_scaled)

    report = classification_report(y_test_local, y_pred_local, output_dict=True, zero_division=0)
    results.append({
        'encoder': encoder_name,
        'model': 'LogisticRegression',
        'accuracy': accuracy_score(y_test_local, y_pred_local),
        'macro_f1': report['macro avg']['f1-score'],
        'class_0_f1': report['0']['f1-score'],
        'class_1_f1': report['1']['f1-score'],
        'precision': precision_score(y_test_local, y_pred_local, zero_division=0),
        'recall': recall_score(y_test_local, y_pred_local, zero_division=0),
        'cv_f1_mean': float(cv_f1_scores.mean()),
        'cv_f1_std': float(cv_f1_scores.std()),
        'best_lr_C': float(lr_search.best_params_['C']) if 'C' in lr_search.best_params_ else np.nan,
        'train_rows': len(df_train_local),
        'test_rows': len(df_test_local),
        'y_test': y_test_local.tolist(),
        'y_pred': y_pred_local.tolist(),
    })

    return results


def expand_embedding_variants(encoder_specs):
    expanded = {}
    for encoder_name, spec in encoder_specs.items():
        base_spec = dict(spec)
        for pooling in POOLING_VARIANTS:
            if base_spec.get('use_global_attention', False):
                for ga in [True, False]:
                    variant = dict(base_spec)
                    variant['pooling'] = pooling
                    variant['use_global_attention'] = ga
                    variant['encoder'] = encoder_name
                    variant_label = f"pool={pooling}|ga={'on' if ga else 'off'}"
                    run_label = f"{encoder_name} | {variant_label}"
                    variant['variant_label'] = variant_label
                    expanded[run_label] = variant
            else:
                variant = dict(base_spec)
                variant['pooling'] = pooling
                variant['encoder'] = encoder_name
                variant_label = f"pool={pooling}"
                run_label = f"{encoder_name} | {variant_label}"
                variant['variant_label'] = variant_label
                expanded[run_label] = variant
    return expanded


datasets = {
    'masked': df_model.copy(),
}

for version_name in datasets:
    datasets[version_name]['target'] = datasets[version_name]['subject_group'].map({'Young_Adults': 0, 'Older_Adults': 1})

# subject split for masked data
train_subjects, test_subjects = subject_level_train_test_split(datasets['masked'])
train_subject_ids = set(train_subjects['subject_id'])
test_subject_ids = set(test_subjects['subject_id'])

results = []
POOLING_VARIANTS = ['cls', 'mean', 'max']
expanded_specs = expand_embedding_variants(ENCODER_SPECS)

for version_name, df_version in datasets.items():
    print(f'\nRunning data version: {version_name}')
    for run_label, spec in expanded_specs.items():
        print(f"  - encoder: {spec['encoder']} | {spec['variant_label']}")
        rows = run_single_experiment(df_version, spec['encoder'], spec, train_subject_ids, test_subject_ids, version_name)
        for row in rows:
            row['data_version'] = version_name
            row['variant'] = spec['variant_label']
            row['run_label'] = f"{run_label} | {row['model']}"
            results.append(row)

# TF-IDF baselines + meta-features
for version_name, df_version in datasets.items():
    df_train_local = df_version[df_version['subject_id'].isin(train_subject_ids)].copy()
    df_test_local = df_version[df_version['subject_id'].isin(test_subject_ids)].copy()

    train_texts = df_train_local['full_conversation'].fillna('').tolist()
    test_texts = df_test_local['full_conversation'].fillna('').tolist()
    train_meta = build_meta_features(df_train_local)
    test_meta = build_meta_features(df_test_local)

    X_train_tfidf, X_test_tfidf = build_tfidf_features(train_texts, test_texts, train_meta, test_meta)
    X_train_bow, X_test_bow = build_bow_features(train_texts, test_texts, train_meta, test_meta)
    y_train_local = df_train_local['target'].values
    y_test_local = df_test_local['target'].values

    train_groups = df_train_local['subject_id'].values
    cv = make_group_cv(train_groups, max_splits=3)
    tfidf_models = {
        'LinearSVC': LinearSVC(random_state=RANDOM_STATE, class_weight='balanced'),
        'LogisticRegression': LogisticRegression(random_state=RANDOM_STATE, max_iter=3000, class_weight='balanced', solver='liblinear'),
    }

    for model_name, clf in tfidf_models.items():
        cv_scores = cross_val_score(clf, X_train_tfidf, y_train_local, cv=cv, groups=train_groups, scoring='f1')
        clf.fit(X_train_tfidf, y_train_local)
        y_pred_local = clf.predict(X_test_tfidf)
        report = classification_report(y_test_local, y_pred_local, output_dict=True, zero_division=0)
        results.append({
            'encoder': 'tfidf',
            'model': model_name,
            'accuracy': accuracy_score(y_test_local, y_pred_local),
            'macro_f1': report['macro avg']['f1-score'],
            'class_0_f1': report['0']['f1-score'],
            'class_1_f1': report['1']['f1-score'],
            'precision': precision_score(y_test_local, y_pred_local, zero_division=0),
            'recall': recall_score(y_test_local, y_pred_local, zero_division=0),
            'cv_f1_mean': float(cv_scores.mean()),
            'cv_f1_std': float(cv_scores.std()),
            'best_lr_C': np.nan,
            'train_rows': len(df_train_local),
            'test_rows': len(df_test_local),
            'y_test': y_test_local.tolist(),
            'y_pred': y_pred_local.tolist(),
            'data_version': version_name,
            'variant': 'word+char+meta',
            'run_label': f"tfidf | word+char+meta | {model_name}",
        })

    for model_name, clf in tfidf_models.items():
        cv_scores = cross_val_score(clf, X_train_bow, y_train_local, cv=cv, groups=train_groups, scoring='f1')
        clf.fit(X_train_bow, y_train_local)
        y_pred_local = clf.predict(X_test_bow)
        report = classification_report(y_test_local, y_pred_local, output_dict=True, zero_division=0)
        results.append({
            'encoder': 'bow',
            'model': model_name,
            'accuracy': accuracy_score(y_test_local, y_pred_local),
            'macro_f1': report['macro avg']['f1-score'],
            'class_0_f1': report['0']['f1-score'],
            'class_1_f1': report['1']['f1-score'],
            'precision': precision_score(y_test_local, y_pred_local, zero_division=0),
            'recall': recall_score(y_test_local, y_pred_local, zero_division=0),
            'cv_f1_mean': float(cv_scores.mean()),
            'cv_f1_std': float(cv_scores.std()),
            'best_lr_C': np.nan,
            'train_rows': len(df_train_local),
            'test_rows': len(df_test_local),
            'y_test': y_test_local.tolist(),
            'y_pred': y_pred_local.tolist(),
            'data_version': version_name,
            'variant': 'word+char+meta',
            'run_label': f"bow | word+char+meta | {model_name}",
        })

results_df = pd.DataFrame(results).sort_values(['data_version', 'macro_f1'], ascending=[True, False]).reset_index(drop=True)
results_df

## 7a) Baseline lock + targeted optimization (to beat current best)

In [ ]:
BASELINE_RUN_LABEL = 'tfidf | word+char+meta | LinearSVC'
BASELINE_DATA_VERSION = 'masked'

baseline_row_df = results_df[
    (results_df['run_label'] == BASELINE_RUN_LABEL)
    & (results_df['data_version'] == BASELINE_DATA_VERSION)
]
if baseline_row_df.empty:
    raise ValueError(f'Baseline row not found for {BASELINE_RUN_LABEL} ({BASELINE_DATA_VERSION}).')

baseline_row = baseline_row_df.iloc[0]
BASELINE_METRICS = {
    'run_label': BASELINE_RUN_LABEL,
    'data_version': BASELINE_DATA_VERSION,
    'macro_f1': float(baseline_row['macro_f1']),
    'accuracy': float(baseline_row['accuracy']),
    'cv_f1_mean': float(baseline_row['cv_f1_mean']) if pd.notna(baseline_row['cv_f1_mean']) else np.nan,
    'cv_f1_std': float(baseline_row['cv_f1_std']) if pd.notna(baseline_row['cv_f1_std']) else np.nan,
}

print('Locked baseline:')
print(BASELINE_METRICS)


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression

def _meta_cols_for_mode(df_in, mode='full'):
    full_cols = [c for c in FEATURE_ENGINEERING_META_COLUMNS if c in df_in.columns]
    if mode == 'none':
        return []
    if mode == 'basic':
        basic = [
            'word_count', 'approx_tokens', 'tokens_longformer', 'tokens_bigbird', 'longformer_excess',
            'msg_to_gpt_char_count', 'msg_to_gpt_word_count', 'msg_from_gpt_char_count', 'msg_from_gpt_word_count',
            'msg_to_gpt_avg_word_len', 'msg_from_gpt_avg_word_len', 'response_word_ratio',
            'msg_to_gpt_sentiment_compound', 'msg_to_gpt_sentiment_pos', 'msg_to_gpt_sentiment_neg',
            'msg_from_gpt_sentiment_compound', 'msg_from_gpt_sentiment_pos', 'msg_from_gpt_sentiment_neg',
        ]
        return [c for c in basic if c in df_in.columns]
    if mode == 'full':
        return full_cols
    raise ValueError("meta_mode must be one of: none, basic, full")

def build_tfidf_features_with_meta_mode(train_texts, test_texts, df_train_local, df_test_local, tfidf_cfg, meta_mode='full'):
    word_vect = TfidfVectorizer(
        ngram_range=tfidf_cfg['word_ngram'],
        min_df=tfidf_cfg['min_df'],
        max_df=tfidf_cfg['max_df'],
        max_features=tfidf_cfg['max_features_word'],
        sublinear_tf=tfidf_cfg['sublinear_tf'],
    )
    char_vect = TfidfVectorizer(
        analyzer='char',
        ngram_range=tfidf_cfg['char_ngram'],
        min_df=tfidf_cfg['min_df'],
        max_features=tfidf_cfg['max_features_char'],
        sublinear_tf=tfidf_cfg['sublinear_tf'],
    )

    X_train_word = word_vect.fit_transform(train_texts)
    X_test_word = word_vect.transform(test_texts)
    X_train_char = char_vect.fit_transform(train_texts)
    X_test_char = char_vect.transform(test_texts)

    X_train_parts = [X_train_word, X_train_char]
    X_test_parts = [X_test_word, X_test_char]

    meta_cols = _meta_cols_for_mode(df_train_local, mode=meta_mode)
    if meta_cols:
        train_meta = (
            df_train_local[meta_cols]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0.0)
            .to_numpy(dtype=float)
        )
        test_meta = (
            df_test_local[meta_cols]
            .apply(pd.to_numeric, errors='coerce')
            .fillna(0.0)
            .to_numpy(dtype=float)
        )
        meta_scaler = StandardScaler(with_mean=False)
        X_train_meta = meta_scaler.fit_transform(csr_matrix(train_meta))
        X_test_meta = meta_scaler.transform(csr_matrix(test_meta))
        X_train_parts.append(X_train_meta)
        X_test_parts.append(X_test_meta)

    X_train = hstack(X_train_parts).tocsr()
    X_test = hstack(X_test_parts).tocsr()
    return X_train, X_test, len(meta_cols)

def evaluate_tfidf_candidate(df_in, train_subject_ids, test_subject_ids, tfidf_cfg, model_kind='linearsvc', model_params=None, meta_mode='full'):
    model_params = model_params or {}
    df_train_local = df_in[df_in['subject_id'].isin(train_subject_ids)].copy()
    df_test_local = df_in[df_in['subject_id'].isin(test_subject_ids)].copy()
    train_texts = df_train_local['full_conversation'].fillna('').tolist()
    test_texts = df_test_local['full_conversation'].fillna('').tolist()

    X_train, X_test, n_meta_cols = build_tfidf_features_with_meta_mode(
        train_texts, test_texts, df_train_local, df_test_local, tfidf_cfg, meta_mode=meta_mode
    )
    y_train_local = df_train_local['target'].values
    y_test_local = df_test_local['target'].values
    train_groups = df_train_local['subject_id'].values
    cv = make_group_cv(train_groups, max_splits=3)

    if model_kind == 'linearsvc':
        clf = LinearSVC(random_state=RANDOM_STATE, class_weight='balanced', **model_params)
    elif model_kind == 'logreg':
        clf = LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', max_iter=4000, **model_params)
    else:
        raise ValueError("model_kind must be 'linearsvc' or 'logreg'")

    cv_scores = cross_val_score(clf, X_train, y_train_local, cv=cv, groups=train_groups, scoring='f1')
    clf.fit(X_train, y_train_local)
    y_pred_local = clf.predict(X_test)
    report = classification_report(y_test_local, y_pred_local, output_dict=True, zero_division=0)

    return {
        'encoder': 'tfidf',
        'model': 'LinearSVC' if model_kind == 'linearsvc' else 'LogisticRegression',
        'variant': f"meta={meta_mode}|w={tfidf_cfg['word_ngram']}|c={tfidf_cfg['char_ngram']}|min_df={tfidf_cfg['min_df']}|max_df={tfidf_cfg['max_df']}|sublinear={tfidf_cfg['sublinear_tf']}",
        'meta_cols_used': n_meta_cols,
        'accuracy': accuracy_score(y_test_local, y_pred_local),
        'macro_f1': report['macro avg']['f1-score'],
        'class_0_f1': report['0']['f1-score'],
        'class_1_f1': report['1']['f1-score'],
        'precision': precision_score(y_test_local, y_pred_local, zero_division=0),
        'recall': recall_score(y_test_local, y_pred_local, zero_division=0),
        'cv_f1_mean': float(cv_scores.mean()),
        'cv_f1_std': float(cv_scores.std()),
    }


In [ ]:
TFIDF_CFGS = [
    {'word_ngram': (1, 2), 'char_ngram': (3, 5), 'min_df': 2, 'max_df': 1.0, 'sublinear_tf': False, 'max_features_word': 30000, 'max_features_char': 30000},
    {'word_ngram': (1, 2), 'char_ngram': (3, 5), 'min_df': 2, 'max_df': 0.95, 'sublinear_tf': True, 'max_features_word': 30000, 'max_features_char': 30000},
    {'word_ngram': (1, 3), 'char_ngram': (3, 6), 'min_df': 2, 'max_df': 0.95, 'sublinear_tf': True, 'max_features_word': 45000, 'max_features_char': 40000},
]

LINEARSVC_PARAMS = [
    {'C': 0.5, 'loss': 'squared_hinge'},
    {'C': 1.0, 'loss': 'squared_hinge'},
    {'C': 2.0, 'loss': 'squared_hinge'},
]
LOGREG_PARAMS = [
    {'C': 0.5, 'solver': 'liblinear', 'penalty': 'l2'},
    {'C': 1.0, 'solver': 'liblinear', 'penalty': 'l2'},
    {'C': 2.0, 'solver': 'liblinear', 'penalty': 'l2'},
]

candidate_rows = []
for cfg in TFIDF_CFGS:
    for meta_mode in ['none', 'basic', 'full']:
        for params in LINEARSVC_PARAMS:
            row = evaluate_tfidf_candidate(df_model, train_subject_ids, test_subject_ids, cfg, model_kind='linearsvc', model_params=params, meta_mode=meta_mode)
            row['params'] = str(params)
            row['tfidf_cfg'] = str(cfg)
            row['model_kind'] = 'linearsvc'
            candidate_rows.append(row)
        for params in LOGREG_PARAMS:
            row = evaluate_tfidf_candidate(df_model, train_subject_ids, test_subject_ids, cfg, model_kind='logreg', model_params=params, meta_mode=meta_mode)
            row['params'] = str(params)
            row['tfidf_cfg'] = str(cfg)
            row['model_kind'] = 'logreg'
            candidate_rows.append(row)

candidate_df = pd.DataFrame(candidate_rows).sort_values(['macro_f1', 'cv_f1_mean'], ascending=False).reset_index(drop=True)

baseline_macro = BASELINE_METRICS['macro_f1']
baseline_cv = BASELINE_METRICS['cv_f1_mean']
candidate_df['beats_baseline_strict'] = (
    (candidate_df['macro_f1'] > baseline_macro)
    & (candidate_df['cv_f1_mean'] >= baseline_cv)
)

print('Top candidate variants (single split):')
candidate_df.head(15)


In [ ]:
import ast

def run_repeated_subject_splits(df_in, top_candidates_df, n_repeats=8, test_size=0.2):
    rows = []
    eval_candidates = top_candidates_df.head(5).to_dict(orient='records')

    for rep in range(n_repeats):
        rs = RANDOM_STATE + rep
        tr_sub, te_sub = subject_level_train_test_split(df_in, test_size=test_size, random_state=rs)
        tr_ids = set(tr_sub['subject_id'])
        te_ids = set(te_sub['subject_id'])

        for cand in eval_candidates:
            variant = cand['variant']
            params = ast.literal_eval(cand['params'])
            cfg = ast.literal_eval(cand['tfidf_cfg'])
            model_kind = cand.get('model_kind', 'linearsvc')
            parts = {kv.split('=')[0]: kv.split('=')[1] for kv in variant.split('|') if '=' in kv}
            meta_mode = parts['meta']

            row = evaluate_tfidf_candidate(
                df_in,
                tr_ids,
                te_ids,
                cfg,
                model_kind=model_kind,
                model_params=params,
                meta_mode=meta_mode,
            )
            row['repeat'] = rep
            row['seed'] = rs
            rows.append(row)

    out = pd.DataFrame(rows)
    summary = out.groupby(['model', 'variant', 'params'], as_index=False).agg(
        macro_f1_mean=('macro_f1', 'mean'),
        macro_f1_std=('macro_f1', 'std'),
        cv_f1_mean=('cv_f1_mean', 'mean'),
        cv_f1_std=('cv_f1_mean', 'std'),
        accuracy_mean=('accuracy', 'mean'),
    ).sort_values(['macro_f1_mean', 'cv_f1_mean'], ascending=False)
    return out, summary.reset_index(drop=True)

repeat_runs_df, repeat_summary_df = run_repeated_subject_splits(df_model, candidate_df, n_repeats=8)
print('Repeated-split robustness summary (top):')
repeat_summary_df.head(10)


## 7b) Optional: MLP classifier (with oversampling)

Reuses cached embeddings and applies RandomOverSampler on the training set to address class imbalance.

In [ ]:
def run_single_experiment_mlp(df_model_input, encoder_name, spec, train_subject_ids, test_subject_ids, data_version):
    df_train_local = df_model_input[df_model_input['subject_id'].isin(train_subject_ids)].copy()
    df_test_local = df_model_input[df_model_input['subject_id'].isin(test_subject_ids)].copy()

    X_train_local, X_test_local = get_cached_embeddings(
        df_train_local,
        df_test_local,
        spec,
        data_version,
    )

    y_train_local = df_train_local['target'].values
    y_test_local = df_test_local['target'].values

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_local)
    X_test_scaled = scaler.transform(X_test_local)

    ros = RandomOverSampler(random_state=RANDOM_STATE)
    X_train_bal, y_train_bal = ros.fit_resample(X_train_scaled, y_train_local)

    mlp = MLPClassifier(
        hidden_layer_sizes=(256,),
        activation='relu',
        solver='adam',
        alpha=1e-4,
        random_state=RANDOM_STATE,
        max_iter=300,
        early_stopping=True,
        n_iter_no_change=10,
    )
    mlp.fit(X_train_bal, y_train_bal)
    y_pred_local = mlp.predict(X_test_scaled)

    report = classification_report(y_test_local, y_pred_local, output_dict=True, zero_division=0)
    return {
        'encoder': encoder_name,
        'model': 'MLPClassifier',
        'accuracy': accuracy_score(y_test_local, y_pred_local),
        'macro_f1': report['macro avg']['f1-score'],
        'class_0_f1': report['0']['f1-score'],
        'class_1_f1': report['1']['f1-score'],
        'precision': precision_score(y_test_local, y_pred_local, zero_division=0),
        'recall': recall_score(y_test_local, y_pred_local, zero_division=0),
        'train_rows': len(df_train_local),
        'test_rows': len(df_test_local),
        'y_test': y_test_local.tolist(),
        'y_pred': y_pred_local.tolist(),
    }


mlp_results = []
MLP_RUN_LABELS = list(expanded_specs.keys())

for version_name, df_version in datasets.items():
    print(f'\nRunning MLP data version: {version_name}')
    for run_label in MLP_RUN_LABELS:
        spec = expanded_specs[run_label]
        print(f"  - encoder: {spec['encoder']} | {spec['variant_label']}")
        row = run_single_experiment_mlp(df_version, spec['encoder'], spec, train_subject_ids, test_subject_ids, version_name)
        row['data_version'] = version_name
        row['variant'] = spec['variant_label']
        row['run_label'] = f"{run_label} | {row['model']}"
        mlp_results.append(row)

mlp_results_df = pd.DataFrame(mlp_results).sort_values(['data_version', 'macro_f1'], ascending=[True, False]).reset_index(drop=True)
mlp_results_df


## 8) Final comparison and summary

In [ ]:
pivot_compare = (
    results_df
    .pivot(index='run_label', columns='data_version', values='accuracy')
    .rename_axis(None, axis=1)
)

print('Accuracy summary (masked only):')
pivot_compare.sort_values(by='masked' if 'masked' in pivot_compare.columns else pivot_compare.columns[0], ascending=False)

In [ ]:
print('Macro F1 summary (masked only):')
f1_compare = results_df.pivot(index='run_label', columns='data_version', values='macro_f1')
f1_compare.sort_values(by='masked' if 'masked' in f1_compare.columns else f1_compare.columns[0], ascending=False)

### Notes
- Classifiers used here include embedding-based Logistic Regression, TF-IDF and BoW baselines (LinearSVC, LogisticRegression), and an optional MLP with oversampling on top of transformer embeddings.
- Tokenizer stage creates input token IDs; embedding stage maps them to dense vectors; classifier predicts class from those vectors.

### Tokenizer length analysis vs embedding extraction (what is the relation?)

1. **Token-length analysis** is a diagnostics step. It only counts how long each conversation becomes under each tokenizer (Longformer / BigBird / LED / Long-T5 / RoBERTa).
2. It helps answer: *will this text fit the model window* (e.g., 4096 for long-context models, 512 for RoBERTa) and how much truncation is expected.
3. **Embedding extraction** is the modeling step: tokenize -> forward pass in transformer -> pooling -> one numeric vector per conversation.
4. For Longformer we try both **global attention on CLS** and **no global attention**, plus CLS/mean/max pooling variants.
5. Those vectors are the features used by the simple linear classifiers and the optional MLP.


In [ ]:
import json
from pathlib import Path

best_bigbird_df = (
    results_df[
        (results_df['data_version'] == 'masked')
        & (results_df['encoder'] == 'bigbird')
        & (results_df['model'] == 'LogisticRegression')
    ]
    .sort_values('macro_f1', ascending=False)
    .head(1)
)

if best_bigbird_df.empty:
    raise ValueError('No masked BigBird LogisticRegression run found in results_df.')

best_bigbird = best_bigbird_df.iloc[0]
variant_parts = {
    kv.split('=')[0]: kv.split('=')[1]
    for kv in str(best_bigbird['variant']).split('|')
    if '=' in kv
}

best_bigbird_payload = {
    'encoder': 'bigbird',
    'model_name': ENCODER_SPECS['bigbird']['model_name'],
    'data_version': 'masked',
    'pooling': variant_parts.get('pool', 'mean'),
    'use_global_attention': variant_parts.get('ga', 'off') == 'on',
    'max_length': int(ENCODER_SPECS['bigbird']['max_length']),
    'best_lr_C': float(best_bigbird['best_lr_C']) if pd.notna(best_bigbird['best_lr_C']) else None,
    'macro_f1': float(best_bigbird['macro_f1']),
    'accuracy': float(best_bigbird['accuracy']),
    'run_label': str(best_bigbird['run_label']),
}

best_bigbird_config_path = Path('best_bigbird_masked_config.json')
best_bigbird_config_path.write_text(json.dumps(best_bigbird_payload, indent=2), encoding='utf-8')
print(f'Saved best BigBird config to: {best_bigbird_config_path.resolve()}')
print(best_bigbird_payload)


def build_mixed_anova_input(df_in, feature_cols, subject_col='subject_id', group_col='subject_group', within_col='TASK', agg_func='mean'):
    required_cols = [subject_col, group_col, within_col]
    missing_base = [c for c in required_cols if c not in df_in.columns]
    if missing_base:
        raise ValueError(f'Missing required columns for mixed ANOVA: {missing_base}')

    missing_features = [c for c in feature_cols if c not in df_in.columns]
    if missing_features:
        raise ValueError(f'Missing selected feature columns: {missing_features}')

    work_df = df_in[required_cols + feature_cols].dropna(subset=[subject_col, group_col, within_col]).copy()
    if agg_func is not None:
        work_df = (
            work_df
            .groupby([subject_col, group_col, within_col], as_index=False)[feature_cols]
            .agg(agg_func)
        )

    return work_df.melt(
        id_vars=[subject_col, group_col, within_col],
        value_vars=feature_cols,
        var_name='feature',
        value_name='value',
    )


def run_mixed_anova_by_feature(anova_long_df, subject_col='subject_id', group_col='subject_group', within_col='TASK'):
    try:
        import pingouin as pg
    except ImportError as exc:
        raise ImportError('pingouin is required for mixed ANOVA. Install with: pip install pingouin') from exc

    outputs = []
    for feature in sorted(anova_long_df['feature'].unique()):
        df_feature = anova_long_df[anova_long_df['feature'] == feature].dropna(subset=['value']).copy()
        if df_feature[within_col].nunique() < 2 or df_feature[group_col].nunique() < 2 or df_feature[subject_col].nunique() < 2:
            continue

        table = pg.mixed_anova(
            data=df_feature,
            dv='value',
            within=within_col,
            between=group_col,
            subject=subject_col,
        )
        table.insert(0, 'feature', feature)
        outputs.append(table)

    if not outputs:
        raise ValueError('No valid features for mixed ANOVA (need variance across groups/tasks/subjects).')

    return pd.concat(outputs, ignore_index=True)


def summarize_mixed_anova(mixed_anova_table: pd.DataFrame) -> pd.DataFrame:
    effect_map = {
        'TASK': 'within_task',
        'subject_group': 'between_group',
        'Interaction': 'interaction',
    }
    compact = mixed_anova_table[['feature', 'Source', 'F', 'p-unc', 'np2']].copy()
    compact['effect'] = compact['Source'].map(effect_map)
    compact = compact[compact['effect'].notna()].copy()

    summary = compact.pivot(index='feature', columns='effect', values=['F', 'p-unc', 'np2'])
    summary.columns = [f'{metric}_{effect}' for metric, effect in summary.columns]
    summary = summary.reset_index()
    summary['sig_interaction'] = np.where(summary['p-unc_interaction'] < 0.05, '*', '')
    summary['sig_between'] = np.where(summary['p-unc_between_group'] < 0.05, '*', '')
    summary['sig_within'] = np.where(summary['p-unc_within_task'] < 0.05, '*', '')
    return summary.sort_values(['p-unc_interaction', 'p-unc_between_group', 'p-unc_within_task'], na_position='last')


MIXED_ANOVA_FEATURES = [
    c for c in df_model.select_dtypes(include=[np.number]).columns
    if c not in {'target'}
]
mixed_anova_input = build_mixed_anova_input(df_model, MIXED_ANOVA_FEATURES)
mixed_anova_results = run_mixed_anova_by_feature(mixed_anova_input)
mixed_anova_summary = summarize_mixed_anova(mixed_anova_results)
print(f'Mixed ANOVA executed for {mixed_anova_summary.shape[0]} numeric features.')
mixed_anova_summary


In [ ]:
results_df

In [ ]:
pivot_compare

Extracting Embeddings: 100%|██████████| 9/9 [00:12<00:00,  1.41s/batch]


In [ ]:
best_rows = results_df.sort_values('macro_f1', ascending=False).head(10)
best_rows


In [ ]:
print('One-shot pipeline complete: masked-only linear runs finished.')
print('Use results_df and pivot_compare as the final artifacts for reporting.')

In [ ]:
results_df[['data_version', 'encoder', 'model', 'variant', 'run_label', 'accuracy', 'macro_f1', 'class_0_f1', 'class_1_f1', 'precision', 'recall', 'cv_f1_mean', 'cv_f1_std', 'best_lr_C']].sort_values(
    ['data_version', 'macro_f1'],
    ascending=[True, False]
).reset_index(drop=True)

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.62      0.72      0.67        18
           1       0.67      0.56      0.61        18

    accuracy                           0.64        36
   macro avg       0.64      0.64      0.64        36
weighted avg       0.64      0.64      0.64        36

Accuracy Score: 0.64


## 9) Evaluation visualizations (F1 and accuracy)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style('whitegrid')

plot_df = results_df.copy()
if 'run_label' not in plot_df.columns:
    plot_df['run_label'] = plot_df['encoder'] + ' | ' + plot_df['data_version']
plot_df = plot_df.sort_values('macro_f1', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors_f1 = plt.cm.RdYlGn(np.linspace(0.25, 0.9, len(plot_df)))
axes[0].barh(plot_df['run_label'], plot_df['macro_f1'], color=colors_f1, edgecolor='black', alpha=0.9)
axes[0].set_title('Macro F1 by Encoder/Variant and Data Version', fontweight='bold')
axes[0].set_xlabel('Macro F1')
axes[0].set_xlim(0, 1)
for i, v in enumerate(plot_df['macro_f1']):
    axes[0].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

colors_acc = plt.cm.Blues(np.linspace(0.35, 0.9, len(plot_df)))
axes[1].barh(plot_df['run_label'], plot_df['accuracy'], color=colors_acc, edgecolor='black', alpha=0.9)
axes[1].set_title('Accuracy by Encoder/Variant and Data Version', fontweight='bold')
axes[1].set_xlabel('Accuracy')
axes[1].set_xlim(0, 1)
for i, v in enumerate(plot_df['accuracy']):
    axes[1].text(v + 0.01, i, f'{v:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
score_cols = ['accuracy', 'macro_f1', 'class_0_f1', 'class_1_f1']
heat_df = results_df.pivot_table(index='run_label', columns='data_version', values='macro_f1')

plt.figure(figsize=(9, 6))
sns.heatmap(heat_df, annot=True, fmt='.3f', cmap='YlGnBu', vmin=0, vmax=1)
plt.title('Macro F1 Heatmap: run label x data version', fontweight='bold')
plt.xlabel('Data version')
plt.ylabel('Run label')
plt.tight_layout()
plt.show()

class_f1_df = (
    results_df[['run_label', 'data_version', 'class_0_f1', 'class_1_f1']]
    .melt(id_vars=['run_label', 'data_version'], value_vars=['class_0_f1', 'class_1_f1'],
          var_name='class_metric', value_name='f1')
)

plt.figure(figsize=(12, 6))
sns.barplot(data=class_f1_df, x='run_label', y='f1', hue='data_version', palette='Set2', ci=None)
plt.title('Per-class F1 by run label (masked only)', fontweight='bold')
plt.ylim(0, 1)
plt.ylabel('F1 score')
plt.xlabel('Run label')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 10) Best-run confusion matrix (mimic clean_only_bert final eval)

In [ ]:
best_idx = results_df['macro_f1'].idxmax()
best_run = results_df.loc[best_idx]
best_name = best_run.get('run_label', f"{best_run['encoder']} | {best_run['data_version']}")

y_true_best = np.array(best_run['y_test'])
y_pred_best = np.array(best_run['y_pred'])

print(f"Best run by Macro F1: {best_name}")
print(f"Macro F1: {best_run['macro_f1']:.4f} | Accuracy: {best_run['accuracy']:.4f}")
print(f"CV F1: {best_run['cv_f1_mean']:.4f} ± {best_run['cv_f1_std']:.4f}")

print('\nClassification report:')
print(classification_report(y_true_best, y_pred_best, target_names=['Young Adults', 'Older Adults'], zero_division=0))

cm = confusion_matrix(y_true_best, y_pred_best)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Young Adults', 'Older Adults'],
            yticklabels=['Young Adults', 'Older Adults'])
axes[0].set_title(f'Confusion Matrix — {best_name}', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Greens', ax=axes[1],
            xticklabels=['Young Adults', 'Older Adults'],
            yticklabels=['Young Adults', 'Older Adults'])
axes[1].set_title(f'Normalized Confusion Matrix — {best_name}', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()